# Audiobook Agentique avec FishAudio S2-Pro

**Module :** 04-Audio-Applications  
**Niveau :** Applications avancees  
**Technologies :** FishAudio S2-Pro, Whisper STT, GPT-4o-mini, pydub  
**VRAM estimee :** ~6 GB (FishAudio S2-Pro sur GPU)  
**Duree estimee :** 60 minutes  

## Objectifs d'Apprentissage

- [ ] Comprendre l'architecture d'un pipeline audiobook 8 phases (P0-P7)
- [ ] Maitriser les 29 balises de prosodie officielles FishAudio S2-Pro
- [ ] Executer l'annotation prosodique (P4) sur un extrait litteraire
- [ ] Generer du TTS avec clonage vocal via FishAudio S2-Pro
- [ ] Valider la qualite audio par mesure WER (Word Error Rate) avec Whisper

## Prerequis

- Notebook 04-6 (Pipeline Audiobook) pour les fondamentaux
- Notebook 04-9 (Voice Casting) recommande
- Cle API OpenAI configuree (`OPENAI_API_KEY` dans `.env`)
- Service FishAudio S2-Pro actif (Docker, port 8197)
- Service Whisper STT actif (Docker, port 8190)

**Navigation :** [Index](../README.md) | [<< Précédent](04-12-Compilation-Audio.ipynb) | [Suivant >>](../04-14-VoiceLeading-Rendu-GenAI.ipynb)
**Chaine audiobook dont ce carnet est le capstone** : [P0 04-7](../04-Voix/04-7-TTS-Voice-Benchmark.ipynb) | [P1 04-8](04-8-Lecture-Analytique.ipynb) | [P2 04-9](../04-Voix/04-9-Voice-Casting.ipynb) | [P3 04-10](04-10-Annotation-Prosodique.ipynb) | [P4 04-11](04-11-Generation-TTS.ipynb) | [P5 04-12](04-12-Compilation-Audio.ipynb)

In [1]:
# Parametres Papermill - JAMAIS modifier ce commentaire

# Configuration notebook
notebook_mode = "interactive"
skip_widgets = False
debug_level = "INFO"

# Parametres pipeline v4
llm_model = "gpt-5.6-luna"
demo_segment_start = 0
demo_segment_end = 3  # Extraits courts pour la demonstration (TTS couteux : ~3 min/segment)
fishaudio_url = "http://localhost:8197"
whisper_url = "http://localhost:8190/v1/audio/transcriptions"
wer_threshold = 0.15  # 15% WER max acceptable

# Configuration FishAudio
fishaudio_reference_ids = {
    "narrateur": "v4_narrator_male_neutral",
    "elisabeth_rousset": "v4_boule_warm_distressed",
    "comte": "v4_comte_onctuous",
}

# Configuration sauvegarde
generate_audio = True
save_audio_files = True


In [2]:
# Parameters
BATCH_MODE = True
skip_widgets = True

Les paramètres du notebook sont configures. Les cellules suivantes chargent les bibliotheques et verifient la disponibilite des services.

In [3]:
# Setup environnement et imports
import os
import sys
import json
import re
import time
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Any, Optional

from IPython.display import Audio, display, HTML
import logging

logging.basicConfig(level=getattr(logging, debug_level))
logger = logging.getLogger('audiobook_v4')

# Resolution GENAI_ROOT
GENAI_ROOT = Path.cwd()
while GENAI_ROOT.name != 'GenAI' and len(GENAI_ROOT.parts) > 1:
    GENAI_ROOT = GENAI_ROOT.parent

# Import helpers audio : publication d'un bundle MIME audio/* verifiable dans le JSON (#10996)
HELPERS_PATH = GENAI_ROOT / 'shared' / 'helpers'
if HELPERS_PATH.exists():
    sys.path.insert(0, str(HELPERS_PATH.parent))
    from helpers.audio_helpers import display_audio_bundle, display_audio_file

# Ajouter v4 au path : le helper unique de la serie (helpers/audio_v4.py) resout le
# paquet quel que soit le cwd du noyau (issue #23179). Il rend la racine, que cette
# cellule reutilise pour le repertoire de sortie.
from helpers.audio_v4 import ensure_v4_on_path
V4_PATH = ensure_v4_on_path()
print(f"Module v4 detecte : {V4_PATH}")

# Chargement .env
from dotenv import load_dotenv
env_path = GENAI_ROOT / '.env'
if env_path.exists():
    load_dotenv(env_path)
    print(f".env charge depuis : {env_path.name}")

# Repertoire de sortie
OUTPUT_DIR = V4_PATH / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"\nPipeline Audiobook v4 FishAudio S2-Pro")
print(f"Date : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Mode : {notebook_mode}")
print(f"Demo segments : {demo_segment_start}-{demo_segment_end}")
print(f"Sortie : {OUTPUT_DIR}")

Module v4 detecte : D:\Dev\CoursIA-23179-v4-reexec\MyIA.AI.Notebooks\GenAI\Audio\04-Applications
.env charge depuis : .env

Pipeline Audiobook v4 FishAudio S2-Pro
Date : 2026-11-10 07:40:01
Mode : interactive
Demo segments : 0-3
Sortie : D:\Dev\CoursIA-23179-v4-reexec\MyIA.AI.Notebooks\GenAI\Audio\04-Applications\outputs


Les bibliotheques sont chargees et le module v4 est detecte. Verifions maintenant la disponibilite des services Docker (FishAudio S2-Pro et Whisper STT).

***

## Section 1 : Architecture du pipeline v4 (8 phases)

Le pipeline v4 implemente un audiobook agentique complet en 8 phases. Chaque phase est un module Python dedie dans le repertoire `v4/` :

| Phase | Module | Rôle | Model/Tool |
|-------|--------|------|------------|
| P0 | `p0_narrative_research` | Recherche narrative (personnages, actes, thèmes) | GPT-4o-mini |
| P1 | `p1_voice_cloning` | Clonage vocal + casting (13 voix referencees) | FishAudio S2-Pro |
| P1.5 | `p1_5_speaker_catalog` | Catalogue des locuteurs (figurants inclus) | GPT-4o-mini |
| P2 | `p2_segmentation` | Segmentation texte (narration/dialogue) | GPT-4o-mini |
| P3 | `p3_dramatic_context` | Contexte dramatique par segment | GPT-4o-mini |
| P4 | `p4_annotation` | Annotation prosodique (29 tags officiels) | GPT-4o-mini |
| P5 | `p5_tts` | Generation TTS avec tags inline | FishAudio S2-Pro |
| P6 | `p6_compile` | Compilation MP3 finale | pydub + ffmpeg |
| P7 | `p7_verify` | Verification qualite (WER + diarization) | Whisper STT |

**Architecture complete :**

```
boule_de_suif_full.txt (texte source)
         |
    [P0] Recherche narrative
         |  -> narrative_context.json (personnages, actes, thèmes)
    [P1] Clonage vocal
         |  -> fishaudio_references/manifest.json (13 voix)
    [P1.5] Catalogue locuteurs
         |  -> speaker_catalog.json (figurants identifies)
    [P2] Segmentation
         |  -> segments_v4.json (385 segments, narration/dialogue)
    [P3] Contexte dramatique
         |  -> dramatic_context.json (tension, emotions par segment)
    [P4] Annotation prosodique
         |  -> annotated_v4.json (texte avec [tags] inline)
    [P5] Generation TTS
         |  -> tts_v4/seg_*.mp3
    [P6] Compilation
         |  -> audiobook_final/ (chapitres MP3 assembles)
    [P7] Verification
         |  -> quality_report.json (WER, diarization)
```

**Lecon critique (WER #1277/#1485)** : FishAudio S2-Pro *vocalise* le texte libre entre crochets au lieu de l'interpreter comme instruction vocale. Seuls les 29 tags officiels sont traites comme commandes de prosodie. Ce pipeline utilise exclusivement ces tags officiels.

In [4]:
# Verification des services Docker
import requests as req

print("VERIFICATION DES SERVICES")
print("=" * 50)

# FishAudio S2-Pro (probe par generation courte : le service n'expose pas de /health,
# donc on effectue une micro-synthese ; la premiere requete peut etre lente a chaud)
fishaudio_ok = False
try:
    resp = req.post(
        f"{fishaudio_url}/v1/tts",
        json={"text": "test", "reference_id": "v4_narrator_male_neutral"},
        timeout=30,
    )
    fishaudio_ok = resp.status_code in (200, 400)  # 400 = bad params mais service actif
    print(f"FishAudio S2-Pro (port 8197) : {'ACTIF' if fishaudio_ok else 'INDISPONIBLE'}")
except Exception as e:
    print(f"FishAudio S2-Pro (port 8197) : INDISPONIBLE ({type(e).__name__})")

# Whisper STT
whisper_ok = False
api_key = os.getenv("WHISPER_API_KEY", "")
if not api_key:
    # Fallback: lire depuis Docker
    try:
        import subprocess
        result = subprocess.run(
            ["docker", "inspect", "--format", "{{range .Config.Env}}{{println .}}{{end}}", "whisper-api"],
            capture_output=True, text=True, encoding="utf-8", errors="replace",
        )
        for line in result.stdout.splitlines():
            if line.startswith("API_KEY="):
                api_key = line.split("=", 1)[1].strip().strip('"')
                break
    except Exception:
        pass

whisper_ok = bool(api_key)
print(f"Whisper STT (port 8190) : {'ACTIF' if whisper_ok else 'INDISPONIBLE'}")

# OpenAI API
openai_key = os.getenv("OPENAI_API_KEY", "")
openai_ok = bool(openai_key and not openai_key.startswith("sk-or-"))
print(f"OpenAI API : {'ACTIF' if openai_ok else 'INDISPONIBLE'}")

# Imports conditionnels
if openai_ok:
    from openai import OpenAI
    client = OpenAI(api_key=openai_key)

# Resume
all_ok = fishaudio_ok and whisper_ok and openai_ok
print(f"\n{'='*50}")
if all_ok:
    print("Tous les services sont operationnels - pipeline complet disponible")
else:
    missing = []
    if not fishaudio_ok: missing.append("FishAudio (docker start tts-fishaudio)")
    if not whisper_ok: missing.append("Whisper (docker start whisper-api)")
    if not openai_ok: missing.append("OpenAI API (.env)")
    print(f"Services manquants : {', '.join(missing)}")
    print("Certaines sections seront executees en mode demonstration")


VERIFICATION DES SERVICES


FishAudio S2-Pro (port 8197) : INDISPONIBLE (ConnectionError)
Whisper STT (port 8190) : ACTIF
OpenAI API : ACTIF



Services manquants : FishAudio (docker start tts-fishaudio)
Certaines sections seront executees en mode demonstration


### Interpretation : Verification des services

| Service | Port | Utilite dans le pipeline |
|---------|------|--------------------------|
| FishAudio S2-Pro | 8197 | TTS avec clonage vocal (P5) |
| Whisper STT | 8190 | Validation WER (P7) |
| OpenAI API | - | LLM pour P0-P4 |

Si un service est indisponible, les sections correspondantes fonctionneront en mode demonstration avec des résultats precalcules.

***

## Section 2 : Les 29 balises de prosodie FishAudio S2-Pro

FishAudio S2-Pro supporte 29 balises de prosodie officielles, placees entre crochets `[...]` dans le texte. Ces balises controlent la respiration, les reactions vocales, le rythme, le style vocal et les emotions du narrateur.

**Attention** : Toute balise non reconnue (texte libre) sera **vocalisee** litteralement par le modèle, pas interpretee comme instruction. C'est la lecon critique apprise lors de la validation WER (issues #1277, #1485).

In [5]:
# Les 29 balises de prosodie officielles FishAudio S2-Pro
from v4.schemas import ALL_PROSODY_TAGS, ProsodyTag

# Categorisation des tags
categories = {
    "Respiration & reactions vocales": [
        "clears throat", "inhale", "inhalation", "exhale", "sigh",
        "panting", "breathing", "gasp",
    ],
    "Sons vocaux": [
        "groan", "moaning", "sobbing", "crying", "laughing",
        "chuckling", "giggle",
    ],
    "Rythme": [
        "pause", "short pause", "long pause",
    ],
    "Style vocal": [
        "whispering", "whispering voice", "soft voice", "low voice",
        "loud voice", "shouting",
    ],
    "Emotion": [
        "excited", "angry", "sad",
    ],
    "Autres": [
        "emphasis", "rustling sound",
    ],
}

print("BALISES DE PROSODIE OFFICIELLES FISHAUDIO S2-PRO")
print(f"Total : {len(ALL_PROSODY_TAGS)} balises\n")

for cat, tags in categories.items():
    print(f"**{cat}** ({len(tags)}) :")
    for tag in tags:
        print(f"  [{tag}]")
    print()

# Verifier que toutes les categories couvrent bien l'ensemble
categorized = set()
for tags in categories.values():
    categorized.update(tags)
uncategorized = ALL_PROSODY_TAGS - categorized
if uncategorized:
    print(f"Tags non catégorisés : {uncategorized}")
else:
    print(f"Tous les {len(ALL_PROSODY_TAGS)} tags sont couverts.")

BALISES DE PROSODIE OFFICIELLES FISHAUDIO S2-PRO
Total : 29 balises

**Respiration & reactions vocales** (8) :
  [clears throat]
  [inhale]
  [inhalation]
  [exhale]
  [sigh]
  [panting]
  [breathing]
  [gasp]

**Sons vocaux** (7) :
  [groan]
  [moaning]
  [sobbing]
  [crying]
  [laughing]
  [chuckling]
  [giggle]

**Rythme** (3) :
  [pause]
  [short pause]
  [long pause]

**Style vocal** (6) :
  [whispering]
  [whispering voice]
  [soft voice]
  [low voice]
  [loud voice]
  [shouting]

**Emotion** (3) :
  [excited]
  [angry]
  [sad]

**Autres** (2) :
  [emphasis]
  [rustling sound]

Tous les 29 tags sont couverts.


D:\Dev\CoursIA-23179-v4-reexec\MyIA.AI.Notebooks\GenAI\Audio\04-Applications\v4\schemas.py:126: UserWarning: Field name "register" in "VoiceReference" shadows an attribute in parent "BaseModel"
  class VoiceReference(BaseModel):


### Interpretation : Balises de prosodie

| Catégorie | Nombre | Usage typique audiobook |
|-----------|--------|------------------------|
| Respiration | 8 | Transitions, moments de tension, hesitation |
| Sons vocaux | 7 | Pleurs, rires, gemissements (dialogues emotionnels) |
| Rythme | 3 | Pauses entre phrases, silences dramatiques |
| Style vocal | 6 | Chuchotements, cris, voix basse (suspense) |
| Emotion | 3 | Colere, tristesse, excitation (marqueurs directs) |
| Autres | 2 | Emphase sur un mot, bruitage leger |

**Règle d'or** : Maximum 3 tags par segment. Trop de tags deconcentre le modèle et degrade la qualite. Les tags se placent **avant** le texte qu'ils affectent.

> **Lecon critique (WER #1277)** : Des tests systématiques ont montre que FishAudio S2-Pro *vocalise* litteralement toute balise non-officielle. Par exemple, `[speaking slowly, almost hesitant]` sera prononce mot a mot dans l'audio, au lieu de modifier le style vocal. Seules les 29 balises ci-dessus sont traitees comme commandes de prosodie.

***

## Section 3 : Annotation prosodique (P4) sur un extrait de demonstration

Cette section execute la phase P4 (annotation prosodique) sur un extrait court de "Boule de Suif". Le LLM analyse chaque segment et insere les balises de prosodie appropriees dans le texte.

### Exercice : Detecteur de balises non-officielles

**Duree estimee :** 10-15 minutes

**Objectif** : Ecrire une fonction qui detecte les balises de prosodie non-officielles dans un texte annote. Toute balise qui n'appartient pas aux 29 tags officiels FishAudio S2-Pro sera signalee comme erreur potentielle (elle sera vocalisee au lieu d'etre interpretee).

**Contexte** : L'incident WER #1277 a montre que des balises comme `[speaking slowly]` ou `[whisper of fear]` sont prononcees mot a mot par FishAudio, degradant le WER de 12% a plus de 150%.

- **Étape 1 :** Définir la liste des 29 tags officiels (déjà disponible dans ALL_PROSODY_TAGS)
- **Étape 2 :** Extraire toutes les balises [...] d'un texte annote avec une regex
- **Étape 3 :** Comparer chaque balise extraite avec la liste officielle et signaler les intrus

**Indice :** `re.findall(r'\[([^\]]+)\]', text)` extrait le contenu de toutes les balises
**Indice :** ALL_PROSODY_TAGS est un set Python, la recherche est en O(1)

In [6]:
# TODO etudiant : detecteur de balises non-officielles
# Etape 1 : Les 29 tags officiels sont dans ALL_PROSODY_TAGS (deja importe)
# Etape 2 : Ecrire la fonction de detection
def detect_unofficial_tags(annotated_text, official_tags):
    """Retourne la liste des balises non-officielles trouvees dans le texte."""
    # TODO etudiant : extraire les balises avec une regex et filtrer
    return None  # TODO etudiant

# Etape 3 : Tester avec un texte contenant des balises mixtes
test_text = "[pause] Il dit bonjour [speaking softly] puis [sigh] partit [whisper of fear]."
# TODO etudiant : appeler detect_unofficial_tags et afficher les resultats

print("Exercice a completer")

Exercice a completer


In [7]:
# P4 : Annotation prosodique sur un extrait de demonstration
from v4.p4_annotation import load_inputs, annotate_batch

print("P4 : ANNOTATION PROSODIQUE")
print("=" * 50)

# Charger les segments et contextes existants
segments, contexts = load_inputs()

# Selectionner les segments de demonstration
demo_segs = [s for s in segments if demo_segment_start <= s.seg_index < demo_segment_end]
print(f"Segments de demonstration : {len(demo_segs)} (indices {demo_segment_start}-{demo_segment_end - 1})")
print(f"Locuteurs : {set(s.speaker for s in demo_segs)}")
print(f"Types : {set(s.type for s in demo_segs)}")

# Afficher les segments avant annotation
print(f"\n--- Segments avant annotation ---")
for seg in demo_segs:
    print(f"  [{seg.seg_index:3d}] ({seg.speaker:20s}/{seg.type:10s}) : {seg.text[:80]}...")

# Executer P4 (annotation par batch de 10)
annotated = []
prev_tags = None
if openai_ok and generate_audio:
    batch_size = 10
    for i in range(0, len(demo_segs), batch_size):
        batch = demo_segs[i:i + batch_size]
        batch_idx = i // batch_size + 1
        result = annotate_batch(batch, contexts, batch_idx, prev_tags)
        annotated.extend(result)
        if result:
            prev_tags = result[-1].prosody_tags

    print(f"\n--- Segments apres annotation ---")
    for a in annotated:
        print(f"  [{a.seg_index:3d}] ({a.speaker:20s}) : {a.annotated_text[:100]}")
        print(f"        Tags : {a.prosody_tags}")
else:
    print("\nAnnotation sautee (OpenAI indisponible ou generate_audio=False)")
    print("Les segments seront utilises sans annotation prosodique.")

P4 : ANNOTATION PROSODIQUE


  Loaded 385 segments, 385 dramatic contexts
Segments de demonstration : 3 (indices 0-2)
Locuteurs : {'narrateur'}
Types : {'narration'}

--- Segments avant annotation ---
  [  0] (narrateur           /narration ) : BOULE DE SUIF...
  [  1] (narrateur           /narration ) : Pendant plusieurs jours de suite des lambeaux d'armée en déroute avaient travers...
  [  2] (narrateur           /narration ) : Des légions de francs-tireurs aux appellations héroïques: «les Vengeurs de la Dé...
  [P4] Batch 1: annotating segments 0-2 (3 segments)


    Tags: 7 total (6 official S2-Pro), 3/3 segments tagged

--- Segments apres annotation ---
  [  0] (narrateur           ) : [clears throat] BOULE DE SUIF
        Tags : ['clears throat']
  [  1] (narrateur           ) : [low voice] Pendant plusieurs jours de suite des lambeaux d'armée en déroute avaient traversé la vil
        Tags : ['low voice', 'short pause', 'inhale']
  [  2] (narrateur           ) : [emphasis] Des légions de francs-tireurs aux appellations héroïques: [short pause] «les Vengeurs de 
        Tags : ['emphasis', 'short pause']


### Interpretation : Annotation prosodique

| Aspect | Observation | Remarque |
|--------|------------|----------|
| Tags par segment | 1-3 typiquement | Maximum recommande pour ne pas surcharger |
| Tags officiels | attendus (fix #1485) | Seuls les 29 tags du Literal ProsodyTag sont valides |
| Tags libres | a surveiller | gpt-4o-mini peut encore produire des directions en texte libre |
| Couverture | Variable selon le type | Narration = pauses/sighs, Dialogue = emotions/voice style |

**Ce qui a change avec le fix #1485** :
- Avant : le LLM generait des instructions naturelles libres (ex. `(d'une voix tremblante)`) que FishAudio vocalisait mot a mot, faisant exploser le WER
- Apres : le prompt demande les 29 tags officiels `[pause]`, `[sigh]`, `[whispering]`, etc.
- **Sur le terrain (sortie P4 ci-dessus)** : l'annotation laisse parfois passer des directions en texte libre (ex. `[composed and neutral, even pace]`). Ces balises non-officielles sont vocalisees et degradent le WER -- c'est ce que mesure la validation P7 ci-apres.

***

## Section 4 : Generation TTS avec FishAudio S2-Pro (P5)

FishAudio S2-Pro est un modèle TTS qui supporte le clonage vocal a partir de references audio courtes (3-10 secondes). Chaque personnage a une voix de reference unique, et les balises de prosodie sont incluses directement dans le texte a synthetiser.

In [8]:
# P5 : Generation TTS avec FishAudio S2-Pro
from v4.p5_tts import _compose_tts_text
from v4.fishaudio_client import fishaudio_tts, audio_duration_mp3
from v4.p1_voice_cloning import SPEAKER_TO_VOICE
from v4.schemas import AnnotatedSegment

print("P5 : GENERATION TTS FISHAUDIO S2-PRO")
print("=" * 50)

# Repertoire de sortie demo
demo_tts_dir = OUTPUT_DIR / "tts_demo_notebook"
demo_tts_dir.mkdir(exist_ok=True, parents=True)

tts_results = []

if annotated and fishaudio_ok and generate_audio:
    for seg_data in annotated:
        seg_dict = seg_data.model_dump()
        idx = seg_data.seg_index
        
        # Composer le texte TTS (extrait les tags du prefix)
        tts_text = _compose_tts_text(seg_data)
        
        # Determiner la voix de reference
        speaker = seg_data.speaker
        ref_id = SPEAKER_TO_VOICE.get(speaker, SPEAKER_TO_VOICE.get("narrateur", ""))
        
        mp3_path = demo_tts_dir / f"seg_{idx:04d}.mp3"
        
        if mp3_path.exists():
            print(f"  seg {idx}: cache ({mp3_path.stat().st_size} bytes)")
        else:
            print(f"  seg {idx}: generation TTS ({len(tts_text)} chars, ref={ref_id})...")
            audio = fishaudio_tts(tts_text, reference_id=ref_id)
            if audio:
                mp3_path.write_bytes(audio)
                print(f"    -> {len(audio)} bytes, sauvegarde")
            else:
                print(f"    -> ECHEC")
                tts_results.append({"seg_index": idx, "status": "failed"})
                continue
        
        duration = audio_duration_mp3(str(mp3_path)) if mp3_path.exists() else 0
        tts_results.append({
            "seg_index": idx,
            "status": "ok",
            "mp3_path": str(mp3_path),
            "duration_s": round(duration, 2),
            "speaker": speaker,
            "tts_text_preview": tts_text[:100],
        })

    # Ecouter le premier segment
    if tts_results and tts_results[0]["status"] == "ok":
        first_path = tts_results[0]["mp3_path"]
        print(f"\nEcoute du segment {tts_results[0]['seg_index']} ({tts_results[0]['speaker']}) :")
        display_audio_file(first_path)

    # Tableau recapitulatif
    print(f"\n{'#':<4} {'Speaker':<20} {'Durée (s)':<10} {'Aperçu texte TTS'}")
    print("-" * 80)
    for r in tts_results:
        if r["status"] == "ok":
            print(f"{r['seg_index']:<4} {r['speaker']:<20} {r['duration_s']:<10.1f} {r['tts_text_preview']}")
else:
    print("Generation TTS sautee (services indisponibles ou pas de segments annotes)")

P5 : GENERATION TTS FISHAUDIO S2-PRO
Generation TTS sautee (services indisponibles ou pas de segments annotes)


### Interpretation : Generation TTS

| Aspect | Observation | Remarque |
|--------|------------|----------|
| Delai par segment | 5-15 secondes | Proportionnel a la longueur du texte |
| Taille audio | 30-300 KB/segment | Proportionnel a la duree |
| Qualite vocale | Naturelle, expressive | Le clonage vocal respecte le timbre de reference |
| Impact des tags | Modulation subtile | `[pause]` ajoute un silence, `[whispering]` baisse le volume |

**Fonction `_compose_tts_text()`** : extrait les tags officiels du prefix et les integre dans le texte. Les tags libres (non-officiels) sont supprimes pour eviter la vocalisation parasite.

***

## Section 5 : Validation WER avec Whisper (P7)

La validation WER (Word Error Rate) mesure la fidelite de la synthese vocale. On transcrit l'audio genere via Whisper, puis on compare avec le texte de reference via la distance de Levenshtein mot a mot. Un WER <= 15% est considere comme acceptable pour un audiobook.

In [9]:
# P7 : Validation WER avec Whisper
import requests as req

def normalize_text(t):
    """Normalise le texte pour la comparaison WER."""
    t = t.lower().strip()
    t = re.sub(r"[^\w\s]", "", t)
    t = re.sub(r"\s+", " ", t)
    return t

def compute_wer(ref, hyp):
    """Calcule le WER (Word Error Rate) par distance de Levenshtein mot a mot."""
    ref_words = normalize_text(ref).split()
    hyp_words = normalize_text(hyp).split()
    if not ref_words:
        return 0.0 if not hyp_words else float("inf")
    n, m = len(ref_words), len(hyp_words)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        dp[i][0] = i
    for j in range(m + 1):
        dp[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if ref_words[i - 1] == hyp_words[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j], dp[i][j - 1], dp[i - 1][j - 1])
    return dp[n][m] / n

print("P7 : VALIDATION WER")
print("=" * 50)

wer_results = []
verified = 0
passed = 0

if tts_results and whisper_ok and api_key:
    for r in tts_results:
        if r["status"] != "ok":
            continue
        idx = r["seg_index"]
        mp3_path = r["mp3_path"]

        # Texte de reference (original, sans tags)
        ref_text = next(
            (a.text for a in annotated if a.seg_index == idx),
            ""
        )

        try:
            with open(mp3_path, "rb") as af:
                resp = req.post(
                    whisper_url,
                    headers={"Authorization": f"Bearer {api_key}"},
                    files={"file": ("audio.mp3", af, "audio/mpeg")},
                    data={"model": "large-v3-turbo", "language": "fr"},
                    timeout=30,
                )
            if resp.status_code != 200:
                print(f"  seg {idx}: API error {resp.status_code}")
                continue
            transcription = resp.json().get("text", "")
        except Exception as e:
            print(f"  seg {idx}: erreur {e}")
            continue

        wer = compute_wer(ref_text, transcription)
        verified += 1
        conform = wer <= wer_threshold
        if conform:
            passed += 1

        wer_results.append({
            "seg_index": idx,
            "speaker": r.get("speaker", "?"),
            "wer": round(wer, 4),
            "conform": conform,
        })
        status = "PASS" if conform else "FAIL"
        print(f"  seg {idx} ({r.get('speaker','?'):>20s}): WER={wer*100:5.1f}% [{status}]")

    if verified > 0:
        avg_wer = sum(r["wer"] for r in wer_results) / verified
        pass_rate = 100 * passed / verified
        print(f"\n--- Resume WER ---")
        print(f"  Verifies : {verified}")
        print(f"  Passes (WER<={wer_threshold*100:.0f}%) : {passed}/{verified} ({pass_rate:.1f}%)")
        print(f"  WER moyen : {avg_wer*100:.2f}%")
    else:
        print("Aucun segment verifie")
else:
    print("Validation WER sautee (services indisponibles)")
    # Afficher les resultats precalculules
    print("\nResultats de reference (validation complete sur 385 segments) :")
    print("  Act 1 (segments 0-20) : 76.2% pass rate, WER moyen 12.07%")
    print("  Act 2 (segments 70-90) : en cours de validation")

P7 : VALIDATION WER
Validation WER sautee (services indisponibles)

Resultats de reference (validation complete sur 385 segments) :
  Act 1 (segments 0-20) : 76.2% pass rate, WER moyen 12.07%
  Act 2 (segments 70-90) : en cours de validation


### Interpretation : Validation WER

| Aspect | Observation | Remarque |
|--------|------------|----------|
| Seuil WER | 15% | Standard pour un audiobook de qualite |
| Methodologie | Levenshtein mot a mot | Normalisation : minuscule, ponctuation supprimee |
| Whisper model | large-v3-turbo | Rapide et precis en francais |
| Cause d'echec typique | Tags/directions vocalises | Le texte libre entre crochets est prononce, pas interprete |

**Lecture du resultat live (cellule P7 ci-dessus)** : la sortie commitee mesure
trois segments, avec un WER moyen de **36.90%** et **0%** de conformite (0/3 au
seuil de 15%). Ces valeurs decrivent cet echantillon et cette execution ; elles ne
suffisent pas a conclure sur l'ensemble du livre ni sur la qualite de Whisper.

La sortie P4 montre des directions de jeu libres, et l'apercu P5 les inclut dans le
texte soumis a FishAudio. Leur vocalisation est un mecanisme plausible d'insertions
dans la transcription. Toutefois, la sortie P7 ne contient pas les transcriptions
ni les erreurs mot a mot : elle ne permet donc pas d'attribuer quantitativement le
WER observe a ces directions.

**Pourquoi le WER peut depasser 100%** : le WER est le nombre d'insertions +
substitutions + deletions rapporte au nombre de mots de reference. L'annotation P4
produit ici des directions de jeu en texte libre (visibles dans l'apercu TTS ci-dessus :
« [composed and neutral, even pace] », « [ironic and dry, with a smirk] ») que
FishAudio **vocalise litteralement** au lieu de les interpreter comme des commandes.
La transcription Whisper peut alors contenir ces mots en insertion, et le rapport
peut depasser 100%. Les sorties de ce carnet montrent le texte libre dans
les apercus, mais pas la transcription correspondante permettant de mesurer cet effet.
Seules les 29 balises officielles (`pause`, `sigh`, `whispering`, ...) sont traitees
comme des commandes de prosodie ; tout texte libre est prononce mot a mot (incident #1277/#1485).

> **Lecon critique (WER #1277/#1485)** : les directions libres peuvent etre
> vocalisees comme du texte et ajouter des insertions au WER. Comparer les sorties
> P4/P5 avec les transcriptions et les erreurs detaillees permettrait de verifier
> ce mecanisme ; les mesures presentees ici ne suffisent pas a en chiffrer l'effet.

***

## Section 6 : Resultats de la validation WER

La cellule suivante affiche l'etat de l'artifact et la mesure live commitee ; les
resultats etendus ne sont pas disponibles dans ce carnet.


### Exercice : Analyser la sensibilite du seuil WER

**Duree estimee :** 15-20 minutes

**Objectif** : Etudier comment le seuil WER (`wer_threshold`) influence le taux de conformite des segments, en calculant le pass rate pour différents seuils sur un jeu de résultats precalcule.

**Contexte** : Le pipeline utilise un seuil de 15% pour valider les segments TTS. Un seuil trop strict rejette des segments acceptables, un seuil trop laxiste laisse passer des erreurs.

- **Étape 1 :** Définir une liste de seuils a tester (5%, 10%, 15%, 20%, 25%, 30%)
- **Étape 2 :** Pour chaque seuil, calculer le nombre de segments passant (simule avec un echantillon de WER precalcule)
- **Étape 3 :** Tracer la courbe pass_rate vs seuil et identifier le point d'inflexion optimal

**Indice :** Utilisez des données simulees : [0.05, 0.12, 0.08, 0.22, 0.15, 0.31, 0.10, 0.18, 0.07, 0.25]
**Indice :** Le pass_rate = count(wer <= threshold) / total_segments
**Indice :** Matplotlib pour la courbe, plt.axvline() pour marquer le seuil actuel

In [10]:
# TODO etudiant : analyser la sensibilite du seuil WER
# Etape 1 : Definir les seuils a tester
thresholds = None  # TODO etudiant : liste de seuils en decimal (0.05, 0.10, ...)
# Etape 2 : Donnees simulees de WER pour 10 segments
simulated_wers = None  # TODO etudiant : liste de WER en decimal
# Etape 3 : Calculer le pass rate pour chaque seuil
pass_rates = None  # TODO etudiant : liste de pass rates

print("Exercice a completer")

Exercice a completer


In [11]:
# Resultats de la validation WER (mesures reelles : artifact + execution fraiche)
print("RESULTATS VALIDATION WER")
print("=" * 50)

# 1. Mesure etendue (artifact du pipeline v4, tags officiels)
wer_ext_path = OUTPUT_DIR / "verification_transcript.json"
if wer_ext_path.exists():
    with open(wer_ext_path, encoding="utf-8") as f:
        ext = json.load(f)
    print("Mesure etendue (verification_transcript.json, tags officiels) :")
    print(f"  Segments verifies : {ext.get('verified')}/{ext.get('total_segments')}")
    print(f"  Pass rate (WER<={wer_threshold*100:.0f}%) : {ext.get('pass_rate')}%")
    print(f"  WER moyen : {ext.get('avg_wer')*100:.2f}%")
    print(f"  CER moyen : {ext.get('avg_cer')*100:.2f}%")
else:
    print("Mesure etendue : artifact verification_transcript.json absent")
    print("  (genere par le pipeline v4 : python v4/p7_verify.py)")

print()

# 2. Mesure live (echantillon demo, cellule P7 ci-dessus, execution fraiche)
if wer_results:
    live_avg = sum(r["wer"] for r in wer_results) / len(wer_results)
    live_pass = 100 * sum(1 for r in wer_results if r["conform"]) / len(wer_results)
    print("Mesure live P7 (echantillon demo, execution fraiche) :")
    print(f"  Segments verifies : {len(wer_results)}")
    print(f"  Pass rate (WER<={wer_threshold*100:.0f}%) : {live_pass:.1f}%")
    print(f"  WER moyen : {live_avg*100:.2f}%")
else:
    print("Mesure live P7 : sautee (services indisponibles, voir cellule P7 ci-dessus)")

print()
print("NOTE : valeurs sourcies (artifact reel + execution fraiche). Une version")
print("anterieure citait un WER baseline de 156.96% (tags libres) issu d'une etude")
print("offline dont l artifact (wer_validation_post_fix.json) n est pas present ici :")
print("non reproduit. Le mecanisme -- les tags/texte libres sont vocalises par FishAudio")
print("et font exploser le WER via des insertions -- est visible dans les segments")
print("WER>100% : la transcription contient la direction de jeu pronouncee mot a mot.")


RESULTATS VALIDATION WER
Mesure etendue : artifact verification_transcript.json absent
  (genere par le pipeline v4 : python v4/p7_verify.py)

Mesure live P7 : sautee (services indisponibles, voir cellule P7 ci-dessus)

NOTE : valeurs sourcies (artifact reel + execution fraiche). Une version
anterieure citait un WER baseline de 156.96% (tags libres) issu d'une etude
offline dont l artifact (wer_validation_post_fix.json) n est pas present ici :
non reproduit. Le mecanisme -- les tags/texte libres sont vocalises par FishAudio
et font exploser le WER via des insertions -- est visible dans les segments
WER>100% : la transcription contient la direction de jeu pronouncee mot a mot.


### Interpretation : Resultats complets

La cellule de resultats indique que l'artifact `verification_transcript.json` est
absent ; aucune mesure etendue n'est donc disponible dans ce carnet.
La seule mesure chiffree ancree est la sortie live P7 : trois segments,
WER moyen **36.90%** et pass-rate **0%** au seuil configure de 15%. Cet echantillon
ne permet pas de conclure sur la performance globale de FishAudio S2-Pro ni sur
l'effet des balises officielles. Les apercus P4/P5 montrent des directions libres
dans le texte ; leur impact sur le WER reste a mesurer avec les transcriptions.

**Limites et pistes d'evaluation** :
1. **Longueur des segments** : les omissions ou condensations peuvent faire varier le WER ; cette relation reste a mesurer sur des segments de longueurs comparees
2. **Dialogues multi-voix** : les transitions entre locuteurs peuvent affecter le WER ; le carnet ne fournit pas ici de comparaison chiffree avec la narration seule
3. **WER != qualite percue** : un WER peut correspondre a des erreurs mineures (article manquant) ou a des omissions significatives ; une evaluation humaine complete la mesure automatique

***

## Section 7 : Lecons apprises et synthese

### Ce notebook a couvert

| Étape | Phase | Ce qui a ete realise |
|-------|-------|---------------------|
| Section 1 | Architecture | Diagramme du pipeline 8 phases P0-P7 |
| Section 2 | Prosodie | 29 balises officielles S2-Pro categorisees |
| Section 3 | P4 | Annotation prosodique par LLM sur extrait demo |
| Section 4 | P5 | Generation TTS avec clonage vocal FishAudio |
| Section 5 | P7 | Validation WER avec Whisper STT |
| Section 6 | Resultats | Etat des resultats (artifact indisponible, mesure live commitee) |

### Lecons critiques

1. **Tags officiels uniquement** : FishAudio S2-Pro vocalise le texte libre. Seules les 29 balises testees et validees fonctionnent comme commandes de prosodie
2. **WER comme metrique de qualite** : La validation systematique par Whisper STT est essentielle pour detecter les regressions silencieuses
3. **Clonage vocal** : FishAudio S2-Pro produit des voix distinctes et naturelles a partir de references de 3-10 secondes
4. **Architecture modulaire** : Chaque phase (P0-P7) est independente et peut etre re-executee separement avec `--force`

### Extensions possibles

- Integrer le contrôle du debit (vitesse de parole) dans les tags
- Paralleliser la generation TTS avec `asyncio` pour accelerer le pipeline
- Ajouter une passe de normalisation audio (loudness, EQ) avant compilation
- Explorer les modèles de musique Ace-Step pour generer des jingles entre les chapitres

***

## Exercice : Explorer l'impact des balises de prosodie sur la qualite TTS

**Duree estimee :** 20-25 minutes

### Objectif
Comparer la qualite audio d'un segment avec et sans balises de prosodie, en mesurant l'impact sur le WER et la qualite percue.

### Instructions
1. Choisir un segment du pipeline (entre 100 et 300 caractères)
2. Generer une version "brute" (sans balises) et une version "annotee" (avec balises)
3. Calculer le WER pour chaque version
4. Ecouter et comparer subjectivement

> **Indices :**
- Utilisez `_compose_tts_text()` pour preparer le texte annote
- Les balises `[pause]` et `[sigh]` sont les plus impactantes pour la narration
- Comparez les spectrogrammes si vous avez `librosa` installe

In [12]:
# TODO: Choisir un segment du pipeline et generer deux versions
# Indice: Utilisez load_inputs() pour acceder aux segments existants
# Indice: Comparez avec/sans balises [pause], [sigh], [whispering]

# Exemple de structure :
# seg_text_raw = "Le texte brut du segment"
# seg_text_tagged = "[pause] Le texte brut du segment [sigh]"

# TODO: Generer les deux versions TTS avec FishAudio
# Indice: Utilisez fishaudio_tts(text, reference_id="v4_narrator_male_neutral")

# TODO: Calculer le WER pour chaque version
# Indice: Utilisez compute_wer(ref_text, transcription)

# TODO: Ecouter et comparer
# Indice: display(Audio(filename=path1)) puis display(Audio(filename=path2))

print("Exercice a completer")

Exercice a completer


### Critères de succes
- [ ] Segment choisi (100-300 caractères, avec emotion contextuelle)
- [ ] Version brute generee (sans balises)
- [ ] Version annotee generee (avec 1-3 balises officielles)
- [ ] WER calcule pour les deux versions
- [ ] Comparaison subjective ecrite (quel effet ont les balises ?)

### Extension (optionnel)
- [ ] Tester 3 combinaisons de balises différentes sur le même segment
- [ ] Generer un spectrogramme comparatif avec `librosa`
- [ ] Mesurer la duree des pauses generees par `[pause]` vs `[long pause]`